<a href="https://colab.research.google.com/github/codedbyumer/urdu-english-nmt/blob/main/urdu_english_nmt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Urdu–English Neural Machine Translation using Low-Resource NMT Techniques

In this project I fine-tune a pretrained multilingual model (`mT5-small`) for
Urdu↔English translation, then try to improve it using LLM-generated synthetic training
data (back-translation and paraphrasing), and compare BLEU / chrF scores before and after
augmentation.

Urdu is spoken by over 230 million people, but it is still a low-resource language in
NLP — there just isn't as much parallel data available compared to languages like French
or German. Instead of training a transformer from scratch (which would need far more data
and compute than I have access to), I fine-tune an existing multilingual model and focus
on how much a low-resource pair like Urdu-English can be improved with data augmentation.

**Pipeline:**
1. Data collection (OPUS-100 via 🤗 Datasets)
2. Preprocessing & train/val/test split
3. Baseline: fine-tune `mT5-small`
4. LLM-based data augmentation (back-translation with NLLB-200, paraphrasing with a small
   open instruct model)
5. Augmented model: fine-tune on baseline + synthetic data
6. Evaluation: BLEU + chrF, baseline vs. augmented
7. Error analysis
8. (Optional) compare against a reference MT system

## 0. Setup

In [1]:
!pip install -q transformers datasets accelerate sentencepiece sacrebleu evaluate sacremoses
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (this will be slow — enable GPU in Colab!)")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 31.7 MB/s eta 0:00:00
CUDA available: True
Device: Tesla T4


### 0b. Saving progress to Google Drive

Colab kept disconnecting on me mid-training, which wipes every variable in memory. Since
some of these steps (training, back-translation, paraphrasing) take 20-60+ minutes, I'm
saving progress to Google Drive after each expensive step. If the runtime restarts, I can
just re-run this cell plus the "resume" cells below instead of redoing everything from
scratch.

In [2]:
import os, json
from google.colab import drive

drive.mount('/content/drive')
CHECKPOINT_DIR = "/content/drive/MyDrive/urdu_nmt_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(name, obj):
    path = os.path.join(CHECKPOINT_DIR, f"{name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)
    print(f"Saved checkpoint: {path}")

def load_checkpoint(name):
    path = os.path.join(CHECKPOINT_DIR, f"{name}.json")
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        print(f"Loaded checkpoint: {path}")
        return data
    return None

print("Checkpoint directory ready:", CHECKPOINT_DIR)
print("Existing checkpoints:", os.listdir(CHECKPOINT_DIR))


Mounted at /content/drive
Checkpoint directory ready: /content/drive/MyDrive/urdu_nmt_checkpoints
Existing checkpoints: []


## 1. Data Collection

I use **OPUS-100** (a curated 100-language subset of OPUS, hosted on 🤗 Hub), which
includes an `en-ur` pair. This gives clean parallel sentences without me having to scrape
OPUS/Tatoeba manually.

For higher quality / domain-specific data, I could swap the `load_dataset(...)` call
below for:
- `Helsinki-NLP/tatoeba_mt` (config `urd-eng`) — smaller, very clean
- The UMC005 corpus (news/religious domain), loaded manually with `pandas.read_csv`

I'm capping the training set at 20k pairs to keep fine-tuning fast on a single Colab GPU,
given my time and compute constraints for this project.

In [3]:
from datasets import load_dataset

N_TRAIN = 20000  # keep small for a solo-project timeline; raise this if you have more GPU time

raw = load_dataset("Helsinki-NLP/opus-100", "en-ur")
print(raw)

# Subsample to keep training fast
train_raw = raw["train"].shuffle(seed=42).select(range(min(N_TRAIN, len(raw["train"]))))
val_raw   = raw["validation"]
test_raw  = raw["test"]

print(f"Train: {len(train_raw)} | Val: {len(val_raw)} | Test: {len(test_raw)}")
print("Example:", train_raw[0])


README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

en-ur/test-00000-of-00001.parquet:   0%|          | 0.00/301k [00:00<?, ?B/s]

en-ur/train-00000-of-00001.parquet:   0%|          | 0.00/148M [00:00<?, ?B/s]

en-ur/validation-00000-of-00001.parquet:   0%|          | 0.00/296k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/753913 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 753913
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})
Train: 20000 | Val: 2000 | Test: 2000
Example: {'translation': {'en': 'And sent them tokens to bring out the best in them.', 'ur': 'ہم نے انہیں وہ نشانیاں عطا فرمائیں جن میں صریح انعام تھا'}}


**Offline fallback:** if I don't have internet access when testing this notebook
(e.g. running it locally without connectivity), I can use this small embedded sample
instead to sanity-check the rest of the pipeline logic.

In [4]:
# --- OFFLINE FALLBACK SAMPLE (only run this cell if the dataset download above fails) ---
# sample_pairs = [
#     {"translation": {"en": "How are you?", "ur": "\u0622\u067e \u06a9\u06cc\u0633\u06d2 \u06c1\u06cc\u06baں\u061f"}},
#     {"translation": {"en": "I am going to school.", "ur": "\u0645\u06cc\u06ba \u0627\u0633\u06a9\u0648\u0644 \u062c\u0627 \u0631\u06c1\u0627 \u06c1\u0648\u06ba\u06d4"}},
#     {"translation": {"en": "This book is very interesting.", "ur": "\u06cc\u06c1 \u06a9\u062a\u0627\u0628 \u0628\u06c1\u062a \u062f\u0644\u0686\u0633\u067e \u06c1\u06d2\u06d4"}},
# ]
# from datasets import Dataset
# train_raw = Dataset.from_list(sample_pairs)
# val_raw = train_raw
# test_raw = train_raw


## 2. Preprocessing

- Clean: strip whitespace, drop empty/duplicate pairs, drop pairs with extreme length
  ratios (a common NMT data-cleaning trick — it removes misaligned sentences).
- Tokenize with the `mT5` tokenizer, framing translation as a text-to-text task with a
  task prefix (`"translate English to Urdu: "`), which is what mT5 expects.

In [5]:
import re

def clean_pairs(dataset, src="en", tgt="ur", min_len=2, max_ratio=3.0):
    cleaned = []
    seen = set()
    for ex in dataset:
        s = ex["translation"][src].strip()
        t = ex["translation"][tgt].strip()
        if len(s) < min_len or len(t) < min_len:
            continue
        key = (s, t)
        if key in seen:
            continue
        seen.add(key)
        ratio = max(len(s), len(t)) / max(1, min(len(s), len(t)))
        if ratio > max_ratio:
            continue
        cleaned.append({"en": s, "ur": t})
    return cleaned

train_pairs = clean_pairs(train_raw)
val_pairs   = clean_pairs(val_raw)
test_pairs  = clean_pairs(test_raw)

print(f"After cleaning -> Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")

# Checkpoint: save cleaned pairs so we don't need to re-download/re-clean after a restart
save_checkpoint("train_pairs", train_pairs)
save_checkpoint("val_pairs", val_pairs)
save_checkpoint("test_pairs", test_pairs)


After cleaning -> Train: 19793 | Val: 1824 | Test: 1835
Saved checkpoint: /content/drive/MyDrive/urdu_nmt_checkpoints/train_pairs.json
Saved checkpoint: /content/drive/MyDrive/urdu_nmt_checkpoints/val_pairs.json
Saved checkpoint: /content/drive/MyDrive/urdu_nmt_checkpoints/test_pairs.json


**If the session restarted on me:** instead of re-running the data-collection and
cleaning cells above, I just run this cell to instantly reload from Google Drive:

In [6]:
# --- RESUME: run this instead of Sections 1-2 if train_pairs/val_pairs/test_pairs
# already exist as checkpoints from a previous run ---
train_pairs = load_checkpoint("train_pairs")
val_pairs = load_checkpoint("val_pairs")
test_pairs = load_checkpoint("test_pairs")
if train_pairs is not None:
    print(f"Resumed -> Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")
else:
    print("No checkpoint found — run Sections 1-2 above from scratch.")


Loaded checkpoint: /content/drive/MyDrive/urdu_nmt_checkpoints/train_pairs.json
Loaded checkpoint: /content/drive/MyDrive/urdu_nmt_checkpoints/val_pairs.json
Loaded checkpoint: /content/drive/MyDrive/urdu_nmt_checkpoints/test_pairs.json
Resumed -> Train: 19793 | Val: 1824 | Test: 1835


In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "google/mt5-small"  # swap to "facebook/mbart-large-50-many-to-many-mmt" for the mBART variant
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

PREFIX = "translate English to Urdu: "
MAX_INPUT_LEN = 64
MAX_TARGET_LEN = 64

def preprocess_batch(examples_en, examples_ur):
    inputs = [PREFIX + e for e in examples_en]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True, padding="max_length")
    labels = tokenizer(text_target=examples_ur, max_length=MAX_TARGET_LEN, truncation=True, padding="max_length")
    # Mask padding tokens in labels with -100 so the loss ignores them.
    # Without this, the model trivially learns to predict pad tokens (loss ~0),
    # which combined with fp16 training reliably produces NaN losses on mT5.
    labels["input_ids"] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

from datasets import Dataset

def to_hf_dataset(pairs):
    ds = Dataset.from_list(pairs)
    ds = ds.map(lambda batch: preprocess_batch(batch["en"], batch["ur"]), batched=True, remove_columns=["en", "ur"])
    return ds

train_ds = to_hf_dataset(train_pairs)
val_ds   = to_hf_dataset(val_pairs[:1000])   # cap val size for speed
test_ds_raw = test_pairs[:500]               # keep raw text version of test set for BLEU/chrF later

print(train_ds)


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Map:   0%|          | 0/19793 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 19793
})


## 3. Baseline Model: Fine-tune `mT5-small`

Standard Seq2Seq fine-tuning with 🤗 `Seq2SeqTrainer`. On a Colab T4 GPU, ~20k pairs for
2-3 epochs should take roughly 30-60 minutes.


In [8]:
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

def load_fresh_model():
    return AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

baseline_model = load_fresh_model()
data_collator = DataCollatorForSeq2Seq(tokenizer, model=baseline_model)

baseline_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-urdu-en-baseline",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-3,
    optim="adafactor",  # mT5 is numerically unstable with AdamW; Adafactor is what it was pretrained with
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    fp16=False,  # mT5 is unstable with fp16 (produces NaN loss); disabled for stability
    logging_steps=100,
    report_to="none",
)

baseline_trainer = Seq2SeqTrainer(
    model=baseline_model,
    args=baseline_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
)

baseline_trainer.train()
baseline_trainer.save_model("./mt5-urdu-en-baseline/final")


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss
1,2.656421,2.638321
2,2.284148,2.405338
3,2.078195,2.329385


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 4. LLM-Based Data Augmentation

Two complementary, fully **open-weight** techniques (no paid API key needed):

1. **Back-translation with NLLB-200-distilled-600M** — I translate extra *English-only*
   sentences into Urdu using Meta's open NLLB model, then use `(generated_ur, original_en)`
   as additional synthetic training pairs. This is the standard low-resource NMT trick for
   getting more training data without needing new human-labeled parallel text.
2. **Paraphrasing with a small open instruct LLM** (`Qwen2.5-1.5B-Instruct`) — I generate
   paraphrases of existing English sentences to add more phrasing diversity to the
   training set without needing new source text.

I load both models directly through 🤗 `transformers` rather than something like Ollama,
since that keeps everything in one self-contained notebook without needing a separate
background server process running.

In [9]:
# --- 4a. Back-translation with NLLB-200-distilled-600M ---
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

N_MONOLINGUAL = 5000  # how many extra English sentences to back-translate

back_translated_pairs = load_checkpoint("back_translated_pairs")

if back_translated_pairs is None:
    # Grab extra monolingual English sentences not already used in train_pairs
    # (reusing the wikitext/opus source as a stand-in for "extra monolingual text")
    extra_en_pool = load_dataset("Helsinki-NLP/opus-100", "en-ur")["train"]
    used_en = set(p["en"] for p in train_pairs)
    extra_en = []
    for ex in extra_en_pool.shuffle(seed=123):
        s = ex["translation"]["en"].strip()
        if s and s not in used_en:
            extra_en.append(s)
        if len(extra_en) >= N_MONOLINGUAL:
            break

    print(f"Collected {len(extra_en)} extra English sentences for back-translation")

    # Note: loading NLLB directly via AutoTokenizer/AutoModelForSeq2SeqLM instead of the
    # `pipeline("translation", ...)` helper — some transformers versions don't register
    # the "translation" pipeline task, so this is the more portable approach.
    nllb_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", src_lang="eng_Latn")
    nllb_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
    nllb_model = nllb_model.to("cuda" if torch.cuda.is_available() else "cpu")

    tgt_lang_id = nllb_tokenizer.convert_tokens_to_ids("urd_Arab")

    BT_BATCH = 32
    back_translated_pairs = []
    for i in range(0, len(extra_en), BT_BATCH):
        batch = extra_en[i:i+BT_BATCH]
        inputs = nllb_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=64).to(nllb_model.device)
        with torch.no_grad():
            generated = nllb_model.generate(
                **inputs,
                forced_bos_token_id=tgt_lang_id,
                max_length=64,
            )
        decoded = nllb_tokenizer.batch_decode(generated, skip_special_tokens=True)
        for src, out in zip(batch, decoded):
            back_translated_pairs.append({"en": src, "ur": out})
        if i % (BT_BATCH * 10) == 0:
            print(f"Back-translated {i+len(batch)}/{len(extra_en)}")
        # Save progress every ~1600 sentences in case of a mid-run disconnect
        if i % (BT_BATCH * 50) == 0 and i > 0:
            save_checkpoint("back_translated_pairs", back_translated_pairs)

    print(f"Generated {len(back_translated_pairs)} synthetic pairs via back-translation")
    print("Example:", back_translated_pairs[0])
    save_checkpoint("back_translated_pairs", back_translated_pairs)
else:
    print(f"Resumed {len(back_translated_pairs)} back-translated pairs from checkpoint — skipping regeneration.")


Collected 5000 extra English sentences for back-translation


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Back-translated 32/5000
Back-translated 352/5000
Back-translated 672/5000
Back-translated 992/5000
Back-translated 1312/5000
Back-translated 1632/5000
Saved checkpoint: /content/drive/MyDrive/urdu_nmt_checkpoints/back_translated_pairs.json
Back-translated 1952/5000
Back-translated 2272/5000
Back-translated 2592/5000
Back-translated 2912/5000
Back-translated 3232/5000
Saved checkpoint: /content/drive/MyDrive/urdu_nmt_checkpoints/back_translated_pairs.json
Back-translated 3552/5000
Back-translated 3872/5000
Back-translated 4192/5000
Back-translated 4512/5000
Back-translated 4832/5000
Saved checkpoint: /content/drive/MyDrive/urdu_nmt_checkpoints/back_translated_pairs.json
Generated 5000 synthetic pairs via back-translation
Example: {'en': "They were pursued by a curse in this world, and so will they be on the Day of Judgement. Lo! 'Ad disbelieved in the Lord. Lo! ruined are 'Ad, the people of Hud.", 'ur': 'اور ان کے پیچھے دنیا میں بھی لعنت ہے اور قیامت کے دن بھی، بیشک عاد نے اپنے رب کا ان

In [10]:
# --- 4b. Paraphrase augmentation with a small open instruct LLM ---
from transformers import pipeline

N_PARAPHRASE = 2000  # how many existing training pairs to paraphrase

paraphrase_pairs = load_checkpoint("paraphrase_pairs")

if paraphrase_pairs is None:
    paraphraser = pipeline(
        "text-generation",
        model="Qwen/Qwen2.5-1.5B-Instruct",
        device=0 if torch.cuda.is_available() else -1,
        max_new_tokens=64,
    )

    def paraphrase_english(sentence):
        messages = [
            {"role": "system", "content": "You paraphrase English sentences. Reply with ONLY the paraphrase, nothing else."},
            {"role": "user", "content": f"Paraphrase this sentence: {sentence}"},
        ]
        prompt = paraphraser.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        out = paraphraser(prompt, do_sample=True, temperature=0.7, top_p=0.9)[0]["generated_text"]
        return out[len(prompt):].strip().split("\n")[0]

    import random
    sample_for_paraphrase = random.sample(train_pairs, min(N_PARAPHRASE, len(train_pairs)))

    paraphrase_pairs = []
    for i, pair in enumerate(sample_for_paraphrase):
        new_en = paraphrase_english(pair["en"])
        if new_en and new_en.lower() != pair["en"].lower():
            # Keep the original Urdu translation paired with the paraphrased English —
            # this teaches the model robustness to phrasing variation.
            paraphrase_pairs.append({"en": new_en, "ur": pair["ur"]})
        if i % 200 == 0:
            print(f"Paraphrased {i}/{len(sample_for_paraphrase)}")
        # Save progress every ~500 sentences in case of a mid-run disconnect
        if i % 500 == 0 and i > 0:
            save_checkpoint("paraphrase_pairs", paraphrase_pairs)

    print(f"Generated {len(paraphrase_pairs)} synthetic pairs via paraphrasing")
    save_checkpoint("paraphrase_pairs", paraphrase_pairs)
else:
    print(f"Resumed {len(paraphrase_pairs)} paraphrase pairs from checkpoint — skipping regeneration.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

Paraphrased 1600/2000


[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

Paraphrased 1800/2000


[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

Generated 1995 synthetic pairs via paraphrasing
Saved checkpoint: /content/drive/MyDrive/urdu_nmt_checkpoints/paraphrase_pairs.json


In [11]:
# --- Combine baseline + synthetic data ---
augmented_pairs = train_pairs + back_translated_pairs + paraphrase_pairs
print(f"Baseline training pairs: {len(train_pairs)}")
print(f"+ Back-translated:       {len(back_translated_pairs)}")
print(f"+ Paraphrased:           {len(paraphrase_pairs)}")
print(f"= Augmented total:       {len(augmented_pairs)}")

augmented_ds = to_hf_dataset(augmented_pairs)


Baseline training pairs: 19793
+ Back-translated:       5000
+ Paraphrased:           1995
= Augmented total:       26788


Map:   0%|          | 0/26788 [00:00<?, ? examples/s]

## 5. Augmented Model: Fine-tune on Baseline + Synthetic Data

Same architecture and hyperparameters as the baseline, just trained on the larger
augmented dataset — this isolates the effect of the augmentation itself, so it's a fair
comparison.

In [17]:
augmented_model = load_fresh_model()  # fresh weights, same base checkpoint, for a fair comparison

augmented_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-urdu-en-augmented",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-3,
    optim="adafactor",  # mT5 is numerically unstable with AdamW; Adafactor is what it was pretrained with
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    fp16=False,  # mT5 is unstable with fp16 (produces NaN loss); disabled for stability
    logging_steps=100,
    report_to="none",
)

augmented_trainer = Seq2SeqTrainer(
    model=augmented_model,
    args=augmented_args,
    train_dataset=augmented_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
)

augmented_trainer.train()
augmented_trainer.save_model("./mt5-urdu-en-augmented/final")


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Epoch,Training Loss,Validation Loss
1,2.427656,2.546147
2,2.084770,2.329658
3,1.857800,2.265646


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 6. Evaluation: BLEU + chrF (Baseline vs. Augmented)

- **BLEU** — standard MT metric, word/n-gram overlap.
- **chrF** — character-level F-score; more informative for morphologically rich
  languages like Urdu, where word-level matching under-rewards close variants.

I evaluate both models on the **same held-out test set** for a fair comparison.

In [18]:
import evaluate
from tqdm import tqdm

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def translate_batch(model, sentences, batch_size=16):
    model.eval()
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    outputs = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer([PREFIX + s for s in batch], return_tensors="pt",
                            padding=True, truncation=True, max_length=MAX_INPUT_LEN).to(model.device)
        with torch.no_grad():
            gen = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=4)
        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        outputs.extend(decoded)
    return outputs

test_en = [p["en"] for p in test_ds_raw]
test_ur = [p["ur"] for p in test_ds_raw]

print("Translating test set with BASELINE model...")
baseline_preds = translate_batch(baseline_model, test_en)

print("Translating test set with AUGMENTED model...")
augmented_preds = translate_batch(augmented_model, test_en)

def score(preds, refs):
    bleu = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    chrf = chrf_metric.compute(predictions=preds, references=[[r] for r in refs])
    return bleu["score"], chrf["score"]

baseline_bleu, baseline_chrf = score(baseline_preds, test_ur)
augmented_bleu, augmented_chrf = score(augmented_preds, test_ur)

print("\n=== Results ===")
print(f"{'Model':<12}{'BLEU':>10}{'chrF':>10}")
print(f"{'Baseline':<12}{baseline_bleu:>10.2f}{baseline_chrf:>10.2f}")
print(f"{'Augmented':<12}{augmented_bleu:>10.2f}{augmented_chrf:>10.2f}")


Translating test set with BASELINE model...


100%|██████████| 32/32 [00:46<00:00,  1.44s/it]


Translating test set with AUGMENTED model...


100%|██████████| 32/32 [00:45<00:00,  1.41s/it]



=== Results ===
Model             BLEU      chrF
Baseline          7.37     25.02
Augmented         8.21     26.10


## 7. Error Analysis

Here I look at concrete examples where the two models disagree — this is the part that
turns the project from "I fine-tuned a model" into an actual analysis: *where and why*
does augmentation help (rare words, idioms, longer sentences)?

In [19]:
import pandas as pd

rows = []
for src, ref, base_pred, aug_pred in zip(test_en, test_ur, baseline_preds, augmented_preds):
    if base_pred != aug_pred:
        rows.append({
            "English (source)": src,
            "Urdu (reference)": ref,
            "Baseline prediction": base_pred,
            "Augmented prediction": aug_pred,
        })

diff_df = pd.DataFrame(rows)
print(f"{len(diff_df)} / {len(test_en)} test sentences differ between models")
diff_df.sample(min(10, len(diff_df)), random_state=1)


457 / 500 test sentences differ between models


,English (source),Urdu (reference),Baseline prediction,Augmented prediction
67,God will not allow to increase whatever illega...,اور جو تم سود دیتے ہو کہ لوگوں کے مال میں افزا...,اور اللہ تعالیٰ تمہارے لئے جو مال تمہارے لئے خ...,اللہ تعالیٰ تمہارے لئے جو کچھ مال تم لوگوں کے ...
146,It is not for the townsfolk of Al-Madinah and ...,مدینہ کے باشندوں اور صحرائی عربوں کے لیے یہ در...,بے شک یہ لوگ مدینہ والوں کے لئے نہیں ہیں اور ی...,اور مدینہ والوں کے لئے یہ مناسب نہیں کہ اللہ ک...
323,God has indeed fulfilled the vision He vouchsa...,بے شک الله نے اپنے رسول کا خواب سچا کر دکھایا ...,اللہ نے اپنے رسول کی نگاہیں پوری کر رکھی ہے کہ...,اللہ نے اپنے رسول کی نگاہیں پوری کر رکھی ہے کہ...
251,Have they not travelled in the land and seen w...,کیا یہ لوگ زمین میں چلے پھرے نہیں کہ دیکھتے کہ...,کیا یہ لوگ زمین میں سیر نہیں کرتے اور دیکھتے ک...,کیا انہوں نے زمین میں سفر نہیں کیا اور دیکھتے ...
224,"But when the truth came to them from Us, they ...",پھر جب ان کے پاس ہمارے حضور سے حق آپہنچا (تو) ...,پھر جب ان کے پاس حق آگیا تو انہوں نے کہا کہ وہ...,پھر جب ان کے پاس ہماری طرف سے حق آگیا تو کہنے ...
407,"This will be followed by a year of rain, and p...",پھر اس کے بعد ایک سال ایسا آئے گا جس میں لوگوں...,یہ ایک سال تک پہنچا دی جائے گی اور لوگ ٹکڑے ہو...,یہ ایک سال کے پیچھے چلنے والی ہے اور لوگ پیٹھ ...
381,The Pharaoh and his people had also received O...,اور البتہ فرعون کے خاندان کے پاس بھی ڈرانے وال...,اور فرعون اور اس کی قوم کے لوگوں نے ہماری تنبی...,فرعون اور اس کی قوم نے بھی ہمارا ڈرانے والا بھیجا
242,And what will show you what is the uphill task?,اورتم کیا جانو یہ گھاٹی کیا ہے,اور تمہیں کیا معلوم ہو جائے گا کہ کس چیز کا ٹھ...,اور تمہیں کیا ہوگیا ہے کہ وہ کھڑے کام کیسا ہے
403,Believe only in those who follow your own reli...,نیز یہ لوگ آپس میں کہتے ہیں کہ اپنے مذہب والے ...,ایمان لانے والوں پر ایمان لاؤ جو اپنے دین کے پ...,ایمان لاؤ وہ لوگ جنہوں نے اپنے دین پر ایمان رک...
92,"The rate of fire, who manufactures what, what ...",اب میں نیو یارک میں جاؤں؟ بہت شکریہ۔ نہیں، ناو...,اور آگ کی طاقت ہے جس نے جو کچھ سال مقرر کیا ہے...,آگ کا وقت ہے جس نے جو کچھ پیدا کیا ہے اور جو ک...


In [20]:
# Simple heuristic error buckets: bucket by source sentence length, to see whether
# augmentation helps more on longer/harder sentences.
diff_df["src_len_words"] = diff_df["English (source)"].apply(lambda s: len(s.split()))
print(diff_df.groupby(pd.cut(diff_df["src_len_words"], bins=[0, 5, 10, 20, 100])).size())


src_len_words
(0, 5]        59
(5, 10]      110
(10, 20]     128
(20, 100]    157
dtype: int64


/tmp/ipykernel_478/1616578279.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(diff_df.groupby(pd.cut(diff_df["src_len_words"], bins=[0, 5, 10, 20, 100])).size())


## 8. (Optional) Compare Against a Reference MT System

If I have a Google Cloud Translation API key, I can contextualize my results against a
production system. This is optional and not required for the core project — I'm skipping
it here since it needs a billing-enabled API key.

In [ ]:
# from google.cloud import translate_v2 as translate
# import os
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "path/to/your/key.json"
# gt_client = translate.Client()
#
# gt_preds = [gt_client.translate(s, target_language="ur")["translatedText"] for s in test_en[:100]]
# gt_bleu, gt_chrf = score(gt_preds, test_ur[:100])
# print(f"Google Translate -> BLEU: {gt_bleu:.2f} | chrF: {gt_chrf:.2f}")


## 9. Summary

- Baseline BLEU / chrF: **7.37 / 25.02**
- Augmented BLEU / chrF: **8.21 / 26.10**
- Dataset size used: 19,793 baseline pairs + 5,000 back-translated pairs + ~1,995
  paraphrased pairs (~26,788 total training pairs for the augmented model)
- Biggest qualitative improvement: augmentation shows the most impact on longer
  sentences — the two models disagree far more often as sentence length increases
  (59 disagreements for 0-5 word sentences, rising to 157 for 20+ word sentences),
  suggesting the extra synthetic data mainly helps with harder, longer translations
  rather than short/simple ones.

Overall, adding LLM-generated synthetic data (back-translation + paraphrasing) improved
both BLEU and chrF over the baseline, with the biggest gains showing up on longer,
harder sentences — which matches what I'd expect from a low-resource setting where the
model hasn't seen enough varied sentence structures during training.